# RouteRanker - Module 2: Feature Engineering and Aggregation

**Goal:** turn raw GTFS-style schedule data into features that let us compare routes.

| Part | You will learn |
|---|---|
| 1 | Why GTFS times (`25:10:00`) are not normal clock times, and how to parse them safely |
| 2 | Hour, minute and time-of-day features |
| 3 | Applying them to a real city file |
| 4 | Day-of-week and weekday-vs-weekend features |
| 5 | The counting trap: rows are not trips |
| 6 | Aggregation with `groupby()`, `nunique()`, `reset_index()` |
| 7 | The same ideas on raw GTFS (`stop_times.txt`, `trips.txt`) |
| 8 | Saving the results |

Run the cells top to bottom (**Shift+Enter**). The heavy lifting lives in `src/routeranker_features.py`; this notebook shows how to *use* it and why each step exists.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

# Works whether VS Code starts the notebook in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

import routeranker_features as rf
from run_module2 import discover_city_files, load_table

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project folder:", ROOT.name, "| pandas", pd.__version__)

ModuleNotFoundError: No module named 'routeranker_features'

## Part 1 - GTFS times are not clock times

In GTFS a service day can last longer than 24 hours. A bus that leaves at 23:50 and reaches its last stop 80 minutes later is written as `25:10:00`, **not** `01:10:00`, because it still belongs to the same service day.

If we hand such text to `pd.to_datetime()`, pandas gives up and returns `NaT` (missing). We would silently lose every after-midnight trip:

In [2]:
print("pd.to_datetime('25:10:00') ->", pd.to_datetime("25:10:00", errors="coerce"))

pd.to_datetime('25:10:00') -> NaT


So we parse the text ourselves into **seconds since the start of the service day**. `parse_gtfs_time` also labels every value:

* `ok` - valid (`08:30:00`, `7:05:00`, `25:10:00`)
* `missing` - blank / `nan` (GTFS allows blank times at middle stops)
* `invalid_format` - not `H:MM:SS` (`abc`, `12:75:00`, `08:30`)
* `suspect_over_limit` - well-formed but hour >= 48 (`86:05:00`): almost certainly an error, so we do not trust it

In [3]:
samples = pd.Series(["08:30:00", "25:10:00", "7:05:00", "", None, "abc", "12:75:00", "86:05:00"], name="raw")
parsed = rf.parse_gtfs_time(samples)
pd.concat([samples, parsed], axis=1)

,raw,secs,status
0,08:30:00,30600.0,ok
1,25:10:00,90600.0,ok
2,7:05:00,25500.0,ok
3,,NaN,missing
4,NaN,NaN,missing
5,abc,NaN,invalid_format
6,12:75:00,NaN,invalid_format
7,86:05:00,NaN,suspect_over_limit


## Part 2 - Hour, minute and time-of-day

`add_time_features` turns one time column into several features. Note the difference between

* `hour` - the **GTFS hour** (can be 24, 25 ...), keeps the "after midnight" information
* `clock_hour` - the hour on a normal 24-hour clock (`hour % 24`), used for the time-of-day period

In [4]:
demo = pd.DataFrame({"scheduled_arrival": ["05:00:00", "08:30:00", "12:05:00", "18:45:00", "23:50:00", "25:10:00", "03:15:00", "abc", None]})
rf.add_time_features(demo, "scheduled_arrival", "arr")

,scheduled_arrival,arr_status,arr_secs,arr_hour,arr_minute,arr_clock_hour,arr_after_midnight,arr_day_offset,arr_period
0,05:00:00,ok,18000,5,0,5,False,0,Morning
1,08:30:00,ok,30600,8,30,8,False,0,Morning
2,12:05:00,ok,43500,12,5,12,False,0,Afternoon
3,18:45:00,ok,67500,18,45,18,False,0,Evening
4,23:50:00,ok,85800,23,50,23,False,0,Night
5,25:10:00,ok,90600,25,10,1,True,1,Night
6,03:15:00,ok,11700,3,15,3,False,0,Night
7,abc,invalid_format,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Unknown
8,NaN,missing,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Unknown


**The time-of-day rules** (applied to the clock hour, so `25:10:00` -> 01:10 -> Night):

In [5]:
rf.PERIOD_TABLE

,period,clock_hours
0,Morning,05:00-11:59
1,Afternoon,12:00-16:59
2,Evening,17:00-20:59
3,Night,21:00-04:59


## Part 3 - A real city

Change `CITY` to try another one (`delhi`, `chennai`, `bengaluru`, `kolkata` ...). Hyderabad is a good first choice: it has after-midnight trips *and* a couple of impossible times.

In [6]:
CITY = "hyderabad"

city_files, _ = discover_city_files(RAW_DIR)
print("Cities found:", sorted(city_files))
raw = load_table(city_files[CITY])
print(CITY, raw.shape)
raw[["Route ID", "gtfs_trips", "stops_on_route", "first_trip_start", "last_trip_start", "service_days"]].head()

Cities found: ['bengaluru', 'chennai', 'delhi', 'hyderabad', 'kolkata', 'mumbai']
hyderabad (2025, 27)


,Route ID,gtfs_trips,stops_on_route,first_trip_start,last_trip_start,service_days
0,1 KOTI MEDICAL COLLEGE-0,2,2,12:45:00,13:20:00,Mon-Sun
1,1-0,131,39,03:40:00,33:20:00,Mon-Sun
2,1-1,172,35,04:10:00,34:00:00,Mon-Sun
3,1/25S-0,35,52,05:30:00,20:45:00,Mon-Sun
4,1/25S-1,34,54,07:15:00,20:10:00,Mon-Sun


In [7]:
routes = rf.engineer_integrated_route_features(raw, CITY)

time_cols = ["route_id", "first_trip_start", "first_start_hour", "first_start_minute", "first_start_period",
             "last_trip_start", "last_start_hour", "last_start_clock_hour", "last_start_after_midnight",
             "last_start_period", "service_span_hours"]
routes[time_cols].head(8)

,route_id,first_trip_start,first_start_hour,first_start_minute,first_start_period,last_trip_start,last_start_hour,last_start_clock_hour,last_start_after_midnight,last_start_period,service_span_hours
0,1 KOTI MEDICAL COLLEGE-0,12:45:00,12,45,Afternoon,13:20:00,13,13,False,Afternoon,0.58
1,1-0,03:40:00,3,40,Night,33:20:00,33,9,True,Morning,29.67
2,1-1,04:10:00,4,10,Night,34:00:00,34,10,True,Morning,29.83
3,1/25S-0,05:30:00,5,30,Morning,20:45:00,20,20,False,Evening,15.25
4,1/25S-1,07:15:00,7,15,Morning,20:10:00,20,20,False,Evening,12.92
5,1/458-0,04:35:00,4,35,Night,04:35:00,4,4,False,Night,0.00
6,1B/D-0,07:05:00,7,5,Morning,07:05:00,7,7,False,Morning,0.00
7,1D/299-0,05:15:00,5,15,Morning,20:20:00,20,20,False,Evening,15.08


Routes whose **last trip starts after midnight** (GTFS hour >= 24). These are valid and must not be dropped:

In [8]:
after_midnight = routes[routes["last_start_after_midnight"].fillna(False)]
print(len(after_midnight), "routes")
after_midnight[time_cols].head()

409 routes


,route_id,first_trip_start,first_start_hour,first_start_minute,first_start_period,last_trip_start,last_start_hour,last_start_clock_hour,last_start_after_midnight,last_start_period,service_span_hours
1,1-0,03:40:00,3,40,Night,33:20:00,33,9,True,Morning,29.67
2,1-1,04:10:00,4,10,Night,34:00:00,34,10,True,Morning,29.83
16,1J-1,21:15:00,21,15,Night,33:35:00,33,9,True,Morning,12.33
34,1Z/251-0,04:35:00,4,35,Night,32:20:00,32,8,True,Morning,27.75
35,1Z/251-1,06:00:00,6,0,Morning,31:10:00,31,7,True,Morning,25.17


How trustworthy are the time values? (`status` counts, and the rows we refused to use)

In [9]:
display(routes[["first_start_status", "last_start_status"]].apply(pd.Series.value_counts).fillna(0).astype(int))
routes[routes["last_start_status"] == "suspect_over_limit"][["route_id", "first_trip_start", "last_trip_start", "last_start_status", "dq_flags"]]

,first_start_status,last_start_status
missing,9,9
ok,2016,2014
suspect_over_limit,0,2


,route_id,first_trip_start,last_trip_start,last_start_status,dq_flags
1166,222P-0,05:50:00,86:05:00,suspect_over_limit,last_time_suspect_48h+;has_suspect_trip_times
1167,222P-1,06:15:00,80:45:00,suspect_over_limit,last_time_suspect_48h+;has_suspect_trip_times


In [10]:
print("Routes by period of first trip:")
routes.loc[routes["has_trips"], "first_start_period"].value_counts().reindex(rf.PERIOD_ORDER).dropna()

Routes by period of first trip:


first_start_period
Morning      1376.0
Afternoon     246.0
Evening       149.0
Night         245.0
Name: count, dtype: float64

## Part 4 - Day of week and weekday vs weekend

The integrated files describe operating days as text such as `Mon-Sun` or `Mon-Fri; Sat; Sun`. We turn that into seven 0/1 flags and a simple `service_type`:

| service_type | Rule |
|---|---|
| All week | runs on all 7 days |
| Weekday only | runs Mon-Fri days, never Sat/Sun |
| Weekend only | runs Sat and/or Sun, never Mon-Fri |
| Weekdays + some weekend | anything in between (e.g. Mon-Sat) |
| Unknown | no day information in the file |

In [11]:
day_cols = ["service_days", "runs_mon", "runs_fri", "runs_sat", "runs_sun", "n_service_days", "runs_weekday", "runs_weekend", "service_type"]
routes[day_cols].drop_duplicates()

,service_days,runs_mon,runs_fri,runs_sat,runs_sun,n_service_days,runs_weekday,runs_weekend,service_type
0,Mon-Sun,1,1,1,1,7,True,True,All week
98,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Unknown
2022,Mon-Fri; Sat; Sun,1,1,1,1,7,True,True,All week


In [12]:
print(routes.loc[routes["has_trips"], "service_type"].value_counts())
# Real calendar dates -> weekday name with pandas (feed validity window, not operating days)
routes[["service_start", "service_start_date", "service_start_day_name", "service_start_is_weekend"]].drop_duplicates().head()

service_type
All week    2016
Name: count, dtype: int64


,service_start,service_start_date,service_start_day_name,service_start_is_weekend
0,20251201,2025-12-01,Monday,False
98,NaN,NaT,NaN,<NA>
2022,20260103,2026-01-03,Saturday,True


## Part 5 - The counting trap

`stop_times.txt` has **one row per (trip, stop)**. A trip with 30 stops owns 30 rows, so counting rows does **not** count trips. Look at a tiny example:

In [13]:
mini_stop_times = pd.DataFrame({
    "trip_id": ["T1"] * 4 + ["T2"] * 3,
    "stop_id": ["S1", "S2", "S3", "S4",   "S1", "S2", "S3"],
})
mini_trips = pd.DataFrame({"trip_id": ["T1", "T2"], "route_id": ["R1", "R1"]})

print("rows in stop_times :", len(mini_stop_times), "  <- NOT the number of trips")
print("distinct trips     :", mini_stop_times["trip_id"].nunique())

joined = mini_stop_times.merge(mini_trips, on="trip_id")          # now every row knows its route
(joined.groupby("route_id")
       .agg(stop_time_rows=("stop_id", "size"),
            distinct_trips=("trip_id", "nunique"),
            distinct_stops=("stop_id", "nunique"))
       .reset_index())

rows in stop_times : 7   <- NOT the number of trips
distinct trips     : 2


,route_id,stop_time_rows,distinct_trips,distinct_stops
0,R1,7,2,4


Route R1 has **2 trips** and **4 stops**, but 7 rows. `groupby()` splits the data into groups, `nunique()` counts distinct values inside each group, `reset_index()` turns the group labels back into a normal column.

## Part 6 - Aggregating the route features

The integrated files already contain the distinct-trip count (`n_trips`, from `gtfs_trips`) and distinct-stop count (`n_stops`, from `stops_on_route`). Now we summarise them.

In [14]:
rf.summarise_by_city(routes)

,city,route_rows,routes_with_trips,total_trips,max_trips_on_one_route,total_stops_on_routes,median_trips_per_route,median_stops_per_route,median_service_span_hours,pct_routes_with_trips
0,hyderabad,2025,2016,51924,1728,57698,4.0,29.0,9.5,99.6


In [15]:
# Your own groupby: how do routes that start in different parts of the day compare?
(routes[routes["has_trips"]]
    .groupby("first_start_period")
    .agg(n_routes=("route_id", "nunique"),
         avg_trips=("n_trips", "mean"),
         avg_stops=("n_stops", "mean"))
    .round(1)
    .reset_index())

,first_start_period,n_routes,avg_trips,avg_stops
0,Afternoon,246,3.4,25.3
1,Evening,149,1.8,25.0
2,Morning,1376,19.5,28.7
3,Night,245,97.8,33.9


In [16]:
rf.period_distribution(routes, "last_start_period")

,city,last_start_period,n_routes,pct_of_city_routes
0,hyderabad,Morning,691,34.3
1,hyderabad,Afternoon,418,20.7
2,hyderabad,Evening,678,33.6
3,hyderabad,Night,227,11.3
4,hyderabad,Unknown,2,0.1


## Part 7 - The same ideas on raw GTFS

If you have the real GTFS zips (e.g. `hyderabad_tgsrtc_bus_gtfs.zip`), put them in `data/raw/gtfs/` and `run_module2.py` will do this automatically. Here we practise on a tiny sample feed that contains every trap on purpose (after-midnight trip, blank time, text instead of a time, a trip with no stop_times, a route that only has `calendar_dates.txt`).

In [17]:
GTFS_ZIP = ROOT / "data" / "raw" / "gtfs" / "sample_city_gtfs.zip"     # swap for a real zip later
if not GTFS_ZIP.exists():
    from make_sample_gtfs import build_sample_gtfs_zip
    build_sample_gtfs_zip(GTFS_ZIP)

tables = rf.read_gtfs_zip(GTFS_ZIP)
{name: (None if df is None else df.shape) for name, df in tables.items()}

{'trips': (8, 4),
 'stop_times': (20, 5),
 'calendar': (2, 10),
 'calendar_dates': (3, 3)}

**Hour and minute from scheduled arrival and departure times** (stop level):

In [18]:
st = tables["stop_times"]
st = rf.add_time_features(st, "arrival_time", "arr")
st = rf.add_time_features(st, "departure_time", "dep")
st[["trip_id", "stop_id", "arrival_time", "arr_status", "arr_hour", "arr_minute", "arr_period", "departure_time", "dep_after_midnight"]]

,trip_id,stop_id,arrival_time,arr_status,arr_hour,arr_minute,arr_period,departure_time,dep_after_midnight
0,T1,S1,08:30:00,ok,8,30,Morning,08:30:00,False
1,T1,S2,08:40:00,ok,8,40,Morning,08:40:00,False
2,T1,S3,08:50:00,ok,8,50,Morning,08:50:00,False
3,T1,S4,09:00:00,ok,9,0,Morning,09:00:00,False
4,T2,S1,17:30:00,ok,17,30,Evening,17:30:00,False
5,T2,S2,17:40:00,ok,17,40,Evening,17:40:00,False
6,T2,S3,17:50:00,ok,17,50,Evening,17:50:00,False
7,T3,S4,06:00:00,ok,6,0,Morning,06:00:00,False
8,T3,S3,06:10:00,ok,6,10,Morning,06:10:00,False
9,T3,S2,06:20:00,ok,6,20,Morning,06:20:00,False


**Trip level** - start = first stop, end = last stop (by `stop_sequence`). Check T4 (past midnight), T5 (blank middle stop), T7 (no stop_times), T8 (`abc`):

In [19]:
trip_features = rf.build_trip_features(tables)
trip_features[["trip_id", "route_id", "start_time_raw", "start_hour", "start_period", "end_time_raw", "end_hour",
               "end_after_midnight", "duration_min", "start_status", "runs_sat"]]

,trip_id,route_id,start_time_raw,start_hour,start_period,end_time_raw,end_hour,end_after_midnight,duration_min,start_status,runs_sat
0,T1,R1,08:30:00,8,Morning,09:00:00,9,False,30.0,ok,0
1,T2,R1,17:30:00,17,Evening,17:50:00,17,False,20.0,ok,0
2,T3,R1,06:00:00,6,Morning,06:20:00,6,False,20.0,ok,0
3,T8,R1,abc,<NA>,Unknown,09:10:00,9,False,NaN,invalid_format,0
4,T4,R2,23:50:00,23,Night,25:10:00,25,True,80.0,ok,1
5,T5,R2,7:05:00,7,Morning,07:40:00,7,False,35.0,ok,1
6,T6,R3,12:00:00,12,Afternoon,12:10:00,12,False,10.0,ok,1
7,T7,R3,NaN,<NA>,Unknown,NaN,<NA>,<NA>,NaN,missing,1


**Route level** - compare `n_trips` / `n_stops` with `n_stop_time_rows`:

In [20]:
route_table = rf.aggregate_routes_from_gtfs(tables, trip_features)
route_table[["route_id", "n_trips", "n_stops", "n_stop_time_rows", "first_trip_start", "last_trip_start",
             "trips_start_morning", "trips_start_evening", "trips_start_night", "trips_start_unknown",
             "trips_ending_after_midnight", "service_type"]]

,route_id,n_trips,n_stops,n_stop_time_rows,first_trip_start,last_trip_start,trips_start_morning,trips_start_evening,trips_start_night,trips_start_unknown,trips_ending_after_midnight,service_type
0,R1,4,4,12,06:00:00,17:30:00,2,1,0,1,0,Weekday only
1,R2,2,3,6,07:05:00,23:50:00,1,0,1,0,1,All week
2,R3,2,2,2,12:00:00,12:00:00,0,0,0,1,0,Weekend only


Expected by hand: R1 has 4 trips / 4 stops / 12 rows, R2 has 2 / 3 / 6, R3 has 2 / 2 / 2.
Note R3: trip T7 has no stop_times, so it is a trip of the route but contributes no stops - an integrated file that says 1 trip would be reconciled as a mismatch.

### Using your real GTFS zip

```python
tables = rf.read_gtfs_zip(ROOT / "data" / "raw" / "gtfs" / "hyderabad_tgsrtc_bus_gtfs.zip")
trip_features = rf.build_trip_features(tables)
route_table = rf.aggregate_routes_from_gtfs(tables, trip_features, use_direction=True)
```
Big feeds can take a minute; that is normal.

## Part 8 - Save the results

`run_module2.py` does this for every city at once. Here we save the city we explored.

In [21]:
out_path = OUT_DIR / f"{CITY}_route_features.csv"
routes.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved", out_path.relative_to(ROOT), "-", routes.shape[0], "rows x", routes.shape[1], "columns")

Saved data/processed/hyderabad_route_features.csv - 2025 rows x 60 columns


## Practice exercises

1. Change `CITY` to `"delhi"`. Why is `service_type` always `Unknown`? (Hint: look at `service_days` in the raw file.)
2. Change `CITY` to `"kolkata"`. Why is `service_span_hours` almost always 0? (Hint: look at `is_frequency_based` and the headway columns.)
3. In `rf.assign_time_period`, change Morning to start at 04:00. Re-run Part 3 and see which counts move.
4. Add a feature `is_peak_start` that is `True` when `first_start_clock_hour` is between 7 and 10.
5. Using `groupby`, find the 10 routes with the most `n_trips`. Are their `n_stops` also high?